In [24]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [25]:
file_path = '../data/processed/master_dataset.csv'

df = pd.read_csv(file_path)

df.head()

,country,year,ev_sales_share,ev_stock,ev_stock_share,charging_points
0,Norway,2024,92.0,960230.0,32.0,31000.0
1,Norway,2023,90.0,900290.0,30.0,28000.0
2,Norway,2022,89.0,790260.0,26.0,25000.0
3,Norway,2021,86.0,630230.0,22.0,19700.0
4,Norway,2020,75.0,490200.0,17.0,17300.0


In [26]:
country_missing = df.groupby('country').agg({
    'ev_stock': lambda x: x.isna().sum(),
    'ev_stock_share': lambda x: x.isna().sum(),
    'charging_points': lambda x: x.isna().sum(),
    'year': 'count'
}).sort_values('ev_stock', ascending=False)

country_missing.head(10)

,ev_stock,ev_stock_share,charging_points,year
country,,,,
Estonia,10,10,10,10
Czech Republic,10,10,10,10
Hungary,10,10,10,10
Slovakia,10,10,10,10
Latvia,10,10,10,10
Ireland,10,10,10,10
Romania,10,10,10,10
Lithuania,9,9,9,9
Slovenia,9,9,9,9


In [27]:
countries_to_remove = [
    'Hungary','Czech Republic','Estonia','Romania',
    'Slovakia','Latvia','Ireland','Bulgaria',
    'Lithuania','Slovenia','Seychelles'
]

df = df[
    ~df['country'].isin(countries_to_remove)
]

Countries with no available observations for the selected explanatory variables (EV stock, EV stock share and charging infrastructure) were excluded from the analysis. Since these variables were entirely absent in the original IEA dataset, reliable imputation was not possible. The remaining missing observations were imputed using country-specific linear interpolation over time, followed by forward and backward filling where necessary.

In [28]:
df.isnull().sum()

country              0
year                 0
ev_sales_share       0
ev_stock            34
ev_stock_share      34
charging_points    171
dtype: int64

In [29]:
master_df = df.sort_values(['country','year'])

for col in ['ev_stock','ev_stock_share','charging_points']:
    master_df[col] = (
        master_df.groupby('country')[col]
        .transform(
            lambda x: x.interpolate()
                       .ffill()
                       .bfill()
        )
    )

In [30]:
master_df[master_df.isnull().any(axis=1)].sort_values(['country', 'year'])

,country,year,ev_sales_share,ev_stock,ev_stock_share,charging_points
409,Croatia,2019,0.41,NaN,NaN,NaN
283,Croatia,2020,1.90,NaN,NaN,NaN
208,Croatia,2021,3.80,NaN,NaN,NaN
183,Croatia,2022,5.00,NaN,NaN,NaN
193,Croatia,2023,4.60,NaN,NaN,NaN
187,Croatia,2024,4.90,NaN,NaN,NaN
466,Cyprus,2019,0.19,NaN,NaN,NaN
311,Cyprus,2020,1.40,NaN,NaN,NaN
292,Cyprus,2021,1.70,NaN,NaN,NaN
174,Cyprus,2022,5.40,NaN,NaN,NaN


In [31]:
master_df = master_df.dropna(subset=['charging_points'])

In [32]:
master_df.isnull().sum()

country            0
year               0
ev_sales_share     0
ev_stock           0
ev_stock_share     0
charging_points    0
dtype: int64

In [33]:
master_df.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)